In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: c:\Users\paula\Documents\Cours\A4\S2\WebData\Project\f1_semantic_web_project


In [1]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

response = requests.post(OLLAMA_URL, json={
    "model": "gemma2:2b",
    "prompt": "What is Formula 1?",
    "stream": False
})

data = response.json()
print(data["response"])

Formula 1, often abbreviated as F1, is the **highest class of international motorsport** for open-wheel racing cars. It's considered the pinnacle of motorsport competition globally and boasts some of the most famous names and highest-stakes races in the world.  

Here are some key aspects of Formula 1:

**The Basics:**

* **Open-Wheel Cars:** F1 cars have an open chassis, allowing for aerodynamic features like wings and diffusers to generate downforce (grip) at high speeds.
* **Teams & Drivers:** Teams design and build the cars, while drivers compete on them in races. Teams are made up of engineers, mechanics, and a variety of support personnel who work together to maximize performance. 
* **Grand Prix Races:** The sport features a calendar of races known as "Grand Prix" held across the globe, with individual countries hosting races each year.  These races involve a complex race circuit design that requires speed, maneuverability, and strategic driving techniques.

**Why is it so excit

In [5]:
from src.rag.pipeline import load_graph, build_schema_summary

g = load_graph("../kg_artifacts/full_kb.ttl")
schema = build_schema_summary(g)
print(schema)

[rag] KB chargé : 42219 triples
PREFIX f1:     <http://example.org/f1/ontology/>
PREFIX entity: <http://example.org/f1/entity/>
PREFIX wd:     <http://www.wikidata.org/entity/>
PREFIX wdt:    <http://www.wikidata.org/prop/direct/>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:    <http://www.w3.org/2002/07/owl#>

# Classes disponibles
- http://example.org/f1/ontology/TeamOrOrganization
- http://example.org/f1/ontology/Driver
- http://example.org/f1/ontology/GrandPrix
- http://example.org/f1/ontology/Season
- http://example.org/f1/ontology/CountryOrPlace
- http://example.org/f1/ontology/Team

# Prédicats disponibles
- http://example.org/f1/ontology/alignmentConfidence
- http://example.org/f1/ontology/isPartOfSeason
- http://example.org/f1/ontology/wonGrandPrix
- http://example.org/f1/ontology/wonConstructorsChampionshipIn

# Exemples de triples
  Cadillac alignmentConfidence 0.6
  Charlie_Whiting alignmentC

In [8]:
from src.rag.pipeline import answer_with_rag, answer_baseline, pretty_print

question = "Which Grand Prix did Lewis Hamilton win?"

print("=== Baseline (sans RAG) ===")
baseline = answer_baseline(question)
print(baseline[:300])

print("\n=== RAG (SPARQL) ===")
result = answer_with_rag(g, schema, question)
pretty_print(result)

=== Baseline (sans RAG) ===
Lewis Hamilton has won the **British Grand Prix** the most times, a total of **8**.  

Let me know if you have any more Formula 1 questions! 😊 🏎️🏆 


=== RAG (SPARQL) ===

Question : Which Grand Prix did Lewis Hamilton win?
Repaired : False

SPARQL:
PREFIX f1: <http://example.org/f1/ontology/>
PREFIX entity: <http://example.org/f1/entity/>
SELECT ?gp WHERE {
    entity:Lewis_Hamilton f1:wonGrandPrix ?gp .
}

Résultats (3 lignes):
gp
http://example.org/f1/entity/Abu_Dhabi_Grand_Prix
http://example.org/f1/entity/Japanese_Grand_Prix
http://example.org/f1/entity/Malaysian_Grand_Prix


In [9]:
questions = [
    "Which team won the Constructors Championship in 2007?",
    "Which season is the Monaco Grand Prix part of?",
    "Which Grand Prix did Ayrton Senna win?",
    "Which Grand Prix is part of the 2024 season?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    
    baseline = answer_baseline(q)
    print(f"\nBaseline: {baseline[:200]}")
    
    result = answer_with_rag(g, schema, q)
    print(f"\nRAG — repaired={result['repaired']} | error={result['error']}")
    print(f"Rows: {len(result['rows'])}")
    if result['rows']:
        print(" | ".join(result['vars']))
        for row in result['rows'][:5]:
            print(" | ".join(row))


Q: Which team won the Constructors Championship in 2007?

Baseline: **Ferrari** won the Constructors' Championship in 2007. 


RAG — repaired=False | error=None
Rows: 0

Q: Which season is the Monaco Grand Prix part of?

Baseline: The Monaco Grand Prix is part of the **Formula 1 World Championship** for every year it's held.  


However, to be specific about which season, you need to know the exact year.  


RAG — repaired=False | error=None
Rows: 14
season
http://example.org/f1/entity/1965
http://example.org/f1/entity/1966
http://example.org/f1/entity/1974
http://example.org/f1/entity/1984
http://example.org/f1/entity/1995

Q: Which Grand Prix did Ayrton Senna win?

Baseline: Ayrton Senna won **3** Formula 1 Grands Prix.

He is widely considered one of the greatest drivers ever, and his victories at these events are all legendary in F1 history.  Which ones you're intereste

RAG — repaired=False | error=None
Rows: 0

Q: Which Grand Prix is part of the 2024 season?

Baseline: The  2024

In [10]:
from rdflib import URIRef

checks = [
    "http://example.org/f1/entity/Ferrari",
    "http://example.org/f1/entity/Ayrton_Senna",
    "http://example.org/f1/entity/2007",
    "http://example.org/f1/entity/2024",
]

for uri in checks:
    triples = list(g.triples((URIRef(uri), None, None)))
    print(f"{uri.split('/')[-1]:20} → {len(triples)} triples")
    for s, p, o in triples[:3]:
        print(f"  {str(p).split('/')[-1]} → {str(o).split('/')[-1]}")

Ferrari              → 30 triples
  22-rdf-syntax-ns#type → Team
  alignmentConfidence → 1.0
  wonConstructorsChampionshipIn → 1950
Ayrton_Senna         → 3 triples
  22-rdf-syntax-ns#type → Driver
  alignmentConfidence → 1.0
  owl#sameAs → Q10490
2007                 → 1 triples
  22-rdf-syntax-ns#type → Season
2024                 → 1 triples
  22-rdf-syntax-ns#type → Season


In [11]:
# vérifie ce qui est disponible dans notre KB
query = """
PREFIX f1: <http://example.org/f1/ontology/>
SELECT ?team ?season WHERE {
    ?team f1:wonConstructorsChampionshipIn ?season .
}
LIMIT 10
"""
for row in g.query(query):
    print(f"{str(row.team).split('/')[-1]:20} → {str(row.season).split('/')[-1]}")

Cadillac             → 1950
Ferrari              → 1950
Cadillac             → 2026
Ferrari              → 2026
McLaren              → 2026
Ferrari              → 1958
Mercedes             → 1958
Ferrari              → 1961
Ferrari              → 1964
Ferrari              → 1966


In [16]:
questions_final = [
    "Which Grand Prix did Lewis Hamilton win?",
    "Which season is the Monaco Grand Prix part of?",
    "Which team won the Constructors Championship in 1958?",
    "Which Grand Prix did Michael Schumacher win?",
    "Which team won the Constructors Championship in 1961?",
]

evaluation = []

for q in questions_final:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    
    baseline = answer_baseline(q)
    result   = answer_with_rag(g, schema, q)
    
    print(f"Baseline : {baseline[:150]}")
    print(f"RAG rows : {len(result['rows'])} | repaired={result['repaired']} | error={result['error']}")
    if result['rows']:
        for row in result['rows'][:3]:
            print(f"  → {' | '.join(r.split('/')[-1] for r in row)}")
    
    evaluation.append({
        "question":       q,
        "baseline":       baseline[:200],
        "rag_rows":       len(result['rows']),
        "rag_correct":    len(result['rows']) > 0,
        "repaired":       result['repaired'],
        "error":          result['error'],
    })

import pandas as pd
eval_df = pd.DataFrame(evaluation)
eval_df.to_csv("../data/rag_evaluation.csv", index=False)
print("\n\nSauvegardé → data/rag_evaluation.csv")
print(eval_df[["question", "rag_rows", "rag_correct", "repaired"]].to_string(index=False))


Q: Which Grand Prix did Lewis Hamilton win?
Baseline : Lewis Hamilton has won the **most** Formula 1 Grand Prixs, but to answer your specific question, he has won **ten** different Grand Prix races in tota
RAG rows : 3 | repaired=False | error=None
  → Abu_Dhabi_Grand_Prix
  → Japanese_Grand_Prix
  → Malaysian_Grand_Prix

Q: Which season is the Monaco Grand Prix part of?
Baseline : The Monaco Grand Prix takes place **during the F1 season** but it's a bit tricky to pin down exactly *which* season.  

Here's why: 

* **The F1 calen
RAG rows : 14 | repaired=False | error=None
  → 1965
  → 1966
  → 1974

Q: Which team won the Constructors Championship in 1958?
Baseline : **Ferrari** won the Constructors' Championship in 1958. 

This was a significant year for Ferrari, as they went on to win the title again in the follo
RAG rows : 2 | repaired=False | error=None
  → Ferrari
  → Mercedes

Q: Which Grand Prix did Michael Schumacher win?
Baseline : Michael Schumacher won the **2004 Australian

In [13]:
# vérifie Michael Schumacher et Ferrari 1958/1961
from rdflib import URIRef

checks = [
    ("Michael_Schumacher", "wonGrandPrix"),
    ("Ferrari", "wonConstructorsChampionshipIn"),
]

for entity, pred in checks:
    query = f"""
    PREFIX f1: <http://example.org/f1/ontology/>
    PREFIX entity: <http://example.org/f1/entity/>
    SELECT ?o WHERE {{
        entity:{entity} f1:{pred} ?o .
    }}
    LIMIT 5
    """
    rows = list(g.query(query))
    print(f"{entity} {pred}:")
    if rows:
        for r in rows:
            print(f"  → {str(r[0]).split('/')[-1]}")
    else:
        print("  → aucun résultat")

Michael_Schumacher wonGrandPrix:
  → aucun résultat
Ferrari wonConstructorsChampionshipIn:
  → 1950
  → 1958
  → 1961
  → 1964
  → 1966


In [15]:
from src.rag.pipeline import generate_sparql

q = "Which team won the Constructors Championship in 1958?"
sparql = generate_sparql(q, schema)
print(sparql)

PREFIX f1: <http://example.org/f1/ontology/>
PREFIX entity: <http://example.org/f1/entity/>
SELECT ?team WHERE {
    ?team f1:wonConstructorsChampionshipIn entity:1958 . 
}


In [17]:
# vérifie quels drivers ont wonGrandPrix dans le KB
query = """
PREFIX f1: <http://example.org/f1/ontology/>
SELECT DISTINCT ?driver WHERE {
    ?driver f1:wonGrandPrix ?gp .
}
"""
drivers = [str(r.driver).split("/")[-1] for r in g.query(query)]
print(f"Drivers avec wonGrandPrix: {len(drivers)}")
print(drivers[:10])

Drivers avec wonGrandPrix: 18
['Alberto_Ascari', 'Sebastian_Vettel', 'Bruce_McLaren', 'Max_Verstappen', 'Carlos_Sainz', 'Monegasque_Charles_Leclerc', 'Oscar_Piastri', 'Carlos_Sainz_Jr', 'Dale_Coyne_Racing227228', 'David_Malukas']


In [18]:
questions_final = [
    "Which Grand Prix did Lewis Hamilton win?",
    "Which season is the Monaco Grand Prix part of?",
    "Which team won the Constructors Championship in 1958?",
    "Which team won the Constructors Championship in 1961?",
    "Which Grand Prix did Max Verstappen win?",
]

evaluation = []

for q in questions_final:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    
    baseline = answer_baseline(q)
    result   = answer_with_rag(g, schema, q)
    
    print(f"Baseline : {baseline[:150]}")
    print(f"RAG rows : {len(result['rows'])} | repaired={result['repaired']} | error={result['error']}")
    if result['rows']:
        for row in result['rows'][:3]:
            print(f"  → {' | '.join(r.split('/')[-1] for r in row)}")
    
    evaluation.append({
        "question":    q,
        "baseline":    baseline[:200],
        "rag_rows":    len(result['rows']),
        "rag_correct": len(result['rows']) > 0,
        "repaired":    result['repaired'],
        "error":       result['error'],
    })

import pandas as pd
eval_df = pd.DataFrame(evaluation)
eval_df.to_csv("../data/rag_evaluation.csv", index=False)
print("\nSauvegardé → data/rag_evaluation.csv")
print(eval_df[["question", "rag_rows", "rag_correct", "repaired"]].to_string(index=False))


Q: Which Grand Prix did Lewis Hamilton win?
Baseline : Lewis Hamilton has won **103** Grand Prixs in his career.  

To answer your question more specifically, you'd need to be more specific about which yea
RAG rows : 3 | repaired=False | error=None
  → Abu_Dhabi_Grand_Prix
  → Japanese_Grand_Prix
  → Malaysian_Grand_Prix

Q: Which season is the Monaco Grand Prix part of?
Baseline : The Monaco Grand Prix is part of the **current** Formula 1 season.  

To be more specific, it's the **2023** Formula 1 season. It takes place during t
RAG rows : 14 | repaired=False | error=None
  → 1965
  → 1966
  → 1974

Q: Which team won the Constructors Championship in 1958?
Baseline : The **Ferrari** team won the Constructors' Championship in 1958. 

RAG rows : 2 | repaired=False | error=None
  → Ferrari
  → Mercedes

Q: Which team won the Constructors Championship in 1961?
Baseline : The **Ferrari** team won the Constructors' Championship in 1961. 🏎️🏆 

RAG rows : 1 | repaired=False | error=None
  → F

In [19]:
import os

for folder in ["../src/rag", "../data"]:
    print(f"\n{folder}/")
    for f in sorted(os.listdir(folder)):
        print(f"  {f}")


../src/rag/
  __init__.py
  __pycache__
  cli.py
  pipeline.py

../data/
  kge
  onthologies
  processed
  rag_evaluation.csv
  raw
